# Frozen A/T/W — decoder-utility diagnostic trên đúng138 ảnh TRAIN
**Internet ON · GPU T4×2 · Save & Run All.** Hai GPU chia69 ảnh mỗi bên; không huấn luyện.
A: gốc. T: chỉnh visual read của cụm về Q phù hợp box đúng. W: về Q của đối tượng khác,
khớp độ lớn thay đổi trước gate/LN. Một mức convex mixing tối đa10%, không quét tham số.
Sinh caption A/T/W và đo lợi ích/selectivity danh từ. GT box chỉ là oracle cho phép thử,
không phải phương pháp inference trên ảnh mới.
**Cần4 dataset:** checkpoint A, prompt-cache .pt gốc, visual-cache, MSCOCO2014 JSON.
Metadata138 ảnh/box/first-noun spans được nhúng từ lượt đã audit; không cần upload lại
review ZIP, JSON prompt, noun-box dataset hay UPT hidden. Input embedding giữ nguyên.

In [ ]:
print('FROZEN A/T/W UTILITY: START',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
OUTPUT=Path('/kaggle/working/routing_utility_ATW_138')
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON):assert p.is_file(),f'Missing input: {p}'
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),'Preserve previous output; use fresh session or output path.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-routing-utility-atw-'))/'repo'
LOG = Path('/tmp/upt_hidden_routing-utility-atw.log')
PROJECT_COMMIT = 'aadedbcc199b9ebaad44f4b0fff0b6fa7d6a156b'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'utility_data.py': '0e6dac99d8b60ced50ca36392dd3c63ec9f73429de3249faeea31fb58903596f', 'utility_intervention.py': '890c9fff45415c27eebcf8bc4cff144726a57eea3eceb1d3e3c57b2e1c4a0c03', 'run_utility.py': 'ad15c1f2a9b197e41d08926bc269f50d3146d3331a8b328811c18a7156312c20', 'test_utility_data.py': 'e114f70a244b2dc0cf0e73ade4539653604f8add3a0c78ffcf965ccd10b1f9ff', 'test_utility_tensors.py': '0c444064176756cc8bd6bcd621779812919108729ae90f24a3634ab33a13ea0e', 'cohort.json': 'b2263515aff870a6247bd97ff41d8c97e924dfadeb506076c5f782bfde197e1c', 'diagnostic_data.py': '15c0eab863ad6500436d25372e34e3395266774838db39b2bdb8e402ff150312', 'run_diagnostic.py': '382653c64ecc7047f908ae717d5d963efa177767ad8db4774093207e286ff731', 'route_probe.py': '7cd9071bc4671873f027c66e4e243e9771947dba8b4ebe83dae4cc7d129783bc', 'audit.py': 'cadf9d172e92e1558020520399ec6013d1141b8ab82c34821d6ba4f0dfe244b4', 'hidden_data.py': '9285d02c5aa7660a5656654e2efe034941bc24d6f7c300958c06fcca6f0d43c7', 'hidden_metric_smoke.py': '8cf64c56b169cdd2407fdeb668408a80d947578632154405c47c7040558a698b', 'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy','Pillow',
    'pycocoevalcap==1.2','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps',
    'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])
CHAIR=Path('/tmp/routing-utility-chair')
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
assert (CHAIR/'chair.pkl').is_file(),'Pinned CHAIR cache missing.'
import torch
assert torch.cuda.device_count()==2,'Select GPU T4x2.'
run([sys.executable,'-u','test_utility_data.py'],cwd=REPO)
run([sys.executable,'-u','test_utility_tensors.py'],cwd=REPO)
run([sys.executable,'-u','hidden_metric_smoke.py',CHAIR],cwd=REPO)


In [ ]:
cfg=dict(checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),chair=str(CHAIR),output=str(OUTPUT),
    cohort_sha256='b2263515aff870a6247bd97ff41d8c97e924dfadeb506076c5f782bfde197e1c',implementation_sha256=SOURCE_HASHES,project_commit=PROJECT_COMMIT)
CONFIG=Path('/tmp/routing_utility_config.json')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
PROTOCOL=OUTPUT/'protocol.json'
try:
    run([sys.executable,'-u','run_utility.py','prepare','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
        '--nproc_per_node=2','--max_restarts=0','run_utility.py','worker','--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_utility.py','evaluate','--config',PROTOCOL],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        for name,source in SOURCES.items():
            target=OUTPUT/'source'/name;target.parent.mkdir(parents=True,exist_ok=True)
            target.write_text(source,encoding='utf-8')
        run([sys.executable,'-u','run_utility.py','archive','--config',CONFIG],cwd=REPO)
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)


Gửi **routing_utility_ATW_138_review.zip**. Không có checkpoint mới.
ZIP khi lỗi chỉ chứa log/source/partial output; receipt ATW_COMPLETE mới xác nhận hoàn tất.
Không tăng10%, đổi pair hoặc bỏ parity nếu FAIL. Nếu khó tìm lỗi, gửi cuối LOG.
Ảnh không can thiệp được giữ caption A trong cả A/T/W, vẫn tính đủ138.
CIDEr dùng TRAIN10k IDF cố định, không so trực tiếp trị số với val/test hoặc dev1000 cũ.
Lượt này chưa chứng minh đóng góp mới hay gain trên ảnh chưa dùng; chỉ kiểm tra có đáng train tiếp không.